In [ ]:
from pathlib import Path

DATA_PATH = Path('data/heart.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('heart.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(
        'Dataset tidak ditemukan. Pastikan heart.csv ada di folder data/heart.csv '
        'atau upload heart.csv ke runtime Colab.'
    )

print(f'Dataset digunakan: {DATA_PATH}')


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
print("Jumlah baris dan kolom:", df.shape)
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
missing_values = df.isnull().sum()

missing_df = pd.DataFrame({
    'Kolom': missing_values.index,
    'Jumlah Missing Value': missing_values.values
})

print("Jumlah Missing Value pada Setiap Kolom:")
display(missing_df)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Jumlah data duplikat:", duplicate_count)

In [ ]:
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Data duplikat berhasil dihapus.")
else:
    print("Tidak terdapat data duplikat.")

In [ ]:
print("Ukuran dataset setelah cleaning:", df.shape)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='target')
plt.title('Distribusi Target Penyakit Jantung')
plt.xlabel('Target: 0 = Tidak Sakit, 1 = Sakit')
plt.ylabel('Jumlah Data')
plt.show()

In [ ]:
df.hist(figsize=(10, 10), bins=20)
plt.suptitle('Distribusi Fitur pada Dataset Heart Disease', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Antar Fitur')
plt.show()

In [ ]:
target_corr = df.corr()['target'].sort_values(ascending=False)

plt.figure(figsize=(8,6))
target_corr.drop('target').sort_values().plot(kind='barh')
plt.title('Korelasi Fitur terhadap Target')
plt.xlabel('Nilai Korelasi')
plt.ylabel('Fitur')
plt.show()

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print("Ukuran fitur X:", X.shape)
print("Ukuran target y:", y.shape)

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns

print("Kolom kategorikal:", list(categorical_cols))

In [ ]:
if len(categorical_cols) > 0:
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    print("Encoding data kategorikal selesai.")
else:
    print("Tidak terdapat kolom kategorikal, encoding tidak diperlukan.")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", X_train.shape[0])
print("Jumlah data testing:", X_test.shape[0])

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

In [ ]:
print("Hasil prediksi Random Forest:")
print(y_pred_rf)

In [ ]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Evaluasi Model Random Forest")
print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1-Score :", f1_rf)

In [ ]:
eval_rf = pd.DataFrame({
    'Model': ['Random Forest'],
    'Accuracy': [accuracy_rf],
    'Precision': [precision_rf],
    'Recall': [recall_rf],
    'F1-Score': [f1_rf]
})

display(eval_rf)

In [ ]:
print("Classification Report Random Forest:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Confusion Matrix Random Forest:")
print(cm_rf)

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix Random Forest')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'Fitur': X.columns,
    'Importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

display(feature_importance)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=feature_importance,
    x='Importance',
    y='Fitur'
)

plt.title('Feature Importance pada Model Random Forest')
plt.xlabel('Nilai Importance')
plt.ylabel('Fitur')
plt.show()

In [ ]:
print("Interpretasi:")
print("Model Random Forest digunakan untuk memprediksi penyakit jantung berdasarkan fitur medis pasien.")
print("Evaluasi dilakukan menggunakan accuracy, precision, recall, dan F1-score.")
print("Confusion matrix digunakan untuk melihat jumlah prediksi benar dan salah.")
print("Feature importance digunakan untuk mengetahui fitur yang paling berpengaruh terhadap hasil prediksi model.")